### Changing the labels to match

In [10]:
import yaml

#Need to change all the .yaml classes to match


#Get the class names from one of the datasets
with open("chess pieces.v3i.yolov11/data.yaml", "r") as f:
    config_1 = yaml.safe_load(f)

classes = config_1["names"]



In [19]:
classes

['BB', 'BK', 'BKN', 'BP', 'BQ', 'BR', 'WB', 'WK', 'WKN', 'WP', 'WQ', 'WR']

In [24]:
#Dataset2
with open("chess.v2i.yolov11/data.yaml", "r") as f:
    config_2 = yaml.safe_load(f)

config_2["names"] = classes

with open("chess.v2i.yolov11/data.yaml", "w") as f:
    yaml.dump(config_2,f,default_flow_style=False)


#Dataset3
with open("Chess.v3i.yolov11/data.yaml", "r") as f:
    config_3 = yaml.safe_load(f)

config_3["names"] = classes

with open("chess.v3i.yolov11/data.yaml", "w") as f:
    yaml.dump(config_3,f,default_flow_style=False)


In [20]:
config_2["names"]

['BB', 'BK', 'BKN', 'BP', 'BQ', 'BR', 'WB', 'WK', 'WKN', 'WP', 'WQ', 'WR']

### Merge the datasets

In [31]:
import supervision as sv
import os

dst_folder = "merged_dataset"
dataset_location_1 = "chess pieces.v3i.yolov11"
dataset_location_2 = "chess.v2i.yolov11"
dataset_location_3 = "chess.v3i.yolov11"



os.makedirs(dst_folder,exist_ok=True)
dataset_names = ["train","test","valid"]



for dataset_name in dataset_names: #For the different data folders open, merge and save
    dataset_1 = sv.DetectionDataset.from_yolo(
        images_directory_path = f"{dataset_location_1}/{dataset_name}/images",
        annotations_directory_path = f"{dataset_location_1}/{dataset_name}/labels",
        data_yaml_path = f"{dataset_location_1}/data.yaml"
    )
    dataset_2 = sv.DetectionDataset.from_yolo(
        images_directory_path = f"{dataset_location_2}/{dataset_name}/images",
        annotations_directory_path = f"{dataset_location_2}/{dataset_name}/labels",
        data_yaml_path = f"{dataset_location_2}/data.yaml"
    )

    dataset_3 = sv.DetectionDataset.from_yolo(
        images_directory_path = f"{dataset_location_3}/{dataset_name}/images",
        annotations_directory_path = f"{dataset_location_3}/{dataset_name}/labels",
        data_yaml_path = f"{dataset_location_3}/data.yaml"
    )

    merged_ds = sv.DetectionDataset.merge([dataset_1,dataset_2,dataset_3])

    os.makedirs(f"{dst_folder}/{dataset_name}/images", exist_ok=True)
    os.makedirs(f"{dst_folder}/{dataset_name}/labels", exist_ok=True)


    merged_ds.as_yolo(
        images_directory_path = f"{dst_folder}/{dataset_name}/images",
        annotations_directory_path = f"{dst_folder}/{dataset_name}/labels",



    )


#Create the final yaml file 
final_yaml = {
    "train": f"{dst_folder}/train/images",
    "val": f"{dst_folder}/valid/images",
    "test": f"{dst_folder}/test/images",
    "nc": len(merged_ds.classes),
    "names": merged_ds.classes
}

with open(f"{dst_folder}/data.yaml", "w") as f:
    yaml.dump(final_yaml, f, default_flow_style=False, sort_keys=False)